# Part 3 — Maps, Space Weather & Ionospheric Confidence Penalty

This notebook wraps up the experiment analysis with four perspectives:

1. **Geographic maps** — all RX stations vs the subset retained by HDBSCAN clustering
2. **Coverage comparison** — bearing coverage before and after HDBSCAN filtering
3. **Space weather** — zoomed timeline of geomagnetic conditions during your runs
4. **Ionospheric confidence penalty** — applied step-by-step to the HDBSCAN F/B estimate

> Run **Part 2** first to understand the HDBSCAN clustering results.
> This notebook re-computes the paired stations and HDBSCAN clusters independently
> from the raw data so you can see the geographic impact.

## 1 · Setup & data loading

In [ ]:
import warnings, micropip
warnings.simplefilter('ignore', DeprecationWarning)
await micropip.install(['plotly', 'jinja2', 'nbformat', 'scikit-learn'])

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from io import StringIO
from pathlib import Path
import math

def _find(name):
    for p in [Path(name), Path('/'+name), Path('files/'+name), Path('/files/'+name)]:
        if p.exists(): return p
    return None

# Experiment spots
csv_path = _find('experiment.csv')
if csv_path is None:
    raise RuntimeError('experiment.csv not found — relaunch from the dashboard.')

df = pd.read_csv(csv_path, dtype='string')
for c in ['snr','distance','distance_miles','distance_km','antenna_azimuth','beacon_bin_azimuth','bearing','rx_lat','rx_lon']:
    if c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')
for c in ['timestamp_utc','time']:
    if c in df.columns: df[c] = pd.to_datetime(df[c], errors='coerce', utc=True)
if 'snr' in df.columns:
    df['snr'] = np.round(df['snr']).astype('Int64').astype('float64')

# QTH metadata for TX position
meta_path = _find('qth_metadata.csv')
tx_lat, tx_lon = np.nan, np.nan
distance_units = 'km'
if meta_path:
    meta = pd.read_csv(meta_path)
    if len(meta):
        row = meta.iloc[0]
        tx_lat = float(row.get('tx_lat', row.get('qth_lat', np.nan)))
        tx_lon = float(row.get('tx_lon', row.get('qth_lon', np.nan)))
        if str(row.get('distance_units','')).strip().lower() == 'mi':
            distance_units = 'mi'
if distance_units == 'mi' and 'distance_miles' in df.columns:
    df['distance'] = df['distance_miles']
elif 'distance_km' in df.columns:
    df['distance'] = df['distance_km']

# Run ledger for roles
run_role_map = {}
rl_path = _find('run_ledger.csv')
if rl_path:
    rl = pd.read_csv(rl_path)
    if {'run_key','role'}.issubset(rl.columns):
        tmp = rl[['run_key','role']].dropna().astype(str)
        tmp['role'] = tmp['role'].str.lower()
        run_role_map = dict(zip(tmp['run_key'], tmp['role']))

# Space weather
sw = None
sw_path = _find('space_weather.csv')
if sw_path:
    sw = pd.read_csv(sw_path)
    if 'timestamp' in sw.columns:
        sw['timestamp'] = pd.to_datetime(sw['timestamp'], errors='coerce', utc=True)
    for c in ['kp_index','solar_flux','sunspot_number','xray_flux','imf_bz']:
        if c in sw.columns: sw[c] = pd.to_numeric(sw[c], errors='coerce')
    print(f'Space weather: {len(sw)} records')
else:
    print('No space_weather.csv found — weather analysis will be skipped.')

print(f'{len(df)} spots loaded')

## 2 · Orientation assignment & paired stations

Quick orientation assignment using the same logic as Part 2, followed by
paired station and HDBSCAN computation so we can use the cluster assignments
throughout this notebook.

In [ ]:
from sklearn.cluster import HDBSCAN
from sklearn.preprocessing import StandardScaler

def angular_distance(a, b):
    return abs((a - b + 180) % 360 - 180)

def classify_orient(frame):
    out = frame.copy()
    out['orientation'] = pd.Series(pd.NA, index=out.index, dtype='string')
    if {'run_key','antenna_azimuth','beacon_bin_azimuth'}.issubset(out.columns):
        rm = (out[out['run_key'].notna()]
              .groupby('run_key',dropna=False)
              .agg(fa=('antenna_azimuth','median'), ba=('beacon_bin_azimuth','median'))
              .reset_index())
        if len(rm):
            rm['orient'] = np.where(
                rm.apply(lambda r: angular_distance(r['fa'],r['ba']), axis=1) <=
                rm.apply(lambda r: angular_distance(r['fa'],(r['ba']+180)%360), axis=1),
                'front','back')
            m = dict(zip(rm['run_key'].astype(str), rm['orient']))
            out['orientation'] = out['run_key'].astype(str).map(m).astype('string')
    miss = out['orientation'].isna()
    if miss.any() and run_role_map and 'run_key' in out.columns:
        out.loc[miss, 'orientation'] = out.loc[miss, 'run_key'].astype(str).map(run_role_map).astype('string')
    miss = out['orientation'].isna()
    if miss.any() and {'bearing','beacon_bin_azimuth'}.issubset(out.columns):
        baz = out.loc[miss,'beacon_bin_azimuth']
        brg = out.loc[miss,'bearing']
        df_ = brg.combine(baz, lambda b,a: angular_distance(b,a))
        db_ = brg.combine(baz, lambda b,a: angular_distance(b,(a+180)%360))
        out.loc[miss, 'orientation'] = np.where(df_ <= db_, 'front', 'back')
    out['orientation'] = out['orientation'].fillna('front').astype('string')
    return out

work = classify_orient(df)
print('Orientation:', work['orientation'].value_counts().to_string())

# Paired stations
def compute_paired_fb(frame, station_col='rxCall'):
    sub = frame[frame['orientation'].isin(['front','back']) & frame['snr'].notna() & frame[station_col].notna()].copy()
    if sub.empty:
        return pd.DataFrame(columns=[station_col,'front_snr','back_snr','fb_delta','spot_count'])
    grouped = sub.groupby([station_col,'orientation'])['snr'].agg(['median','count']).unstack('orientation')
    if ('median','front') not in grouped.columns or ('median','back') not in grouped.columns:
        return pd.DataFrame(columns=[station_col,'front_snr','back_snr','fb_delta','spot_count'])
    paired = grouped.dropna(subset=[('median','front'), ('median','back')]).copy()
    if paired.empty:
        return pd.DataFrame(columns=[station_col,'front_snr','back_snr','fb_delta','spot_count'])
    result = pd.DataFrame({
        station_col: paired.index,
        'front_snr': paired[('median','front')].values,
        'back_snr': paired[('median','back')].values,
        'front_spots': paired[('count','front')].fillna(0).astype(int).values,
        'back_spots': paired[('count','back')].fillna(0).astype(int).values,
    })
    result['fb_delta'] = result['front_snr'] - result['back_snr']
    result['spot_count'] = result['front_spots'] + result['back_spots']
    return result.sort_values('fb_delta', ascending=False).reset_index(drop=True)

pairs = compute_paired_fb(work)
total_rx = work['rxCall'].nunique() if 'rxCall' in work.columns else 0
print(f'\nTotal unique RX stations: {total_rx}')
print(f'Paired stations (heard in BOTH front & back): {len(pairs)}')
print(f'Unpaired stations (heard in only one orientation): {total_rx - len(pairs)}')

# HDBSCAN clustering
def hodges_lehmann(values):
    vals = np.asarray(values, dtype=float)
    n = len(vals)
    if n <= 1: return float(vals[0]) if n == 1 else np.nan
    pw = [(vals[i] + vals[j]) / 2.0 for i in range(n) for j in range(i, n)]
    return float(np.median(pw))

hdbscan_stations = set()
cluster_scores = []
cluster_points = []

if len(pairs) >= 4:
    station_col = 'rxCall'
    points = []
    for _, row in pairs.iterrows():
        call = row[station_col]
        stn_rows = work[work[station_col] == call]
        bearing = stn_rows['bearing'].median() if 'bearing' in stn_rows.columns else 0
        distance = stn_rows['distance'].median() if 'distance' in stn_rows.columns else 0
        bearing_rad = np.radians(bearing)
        points.append({
            'station': call,
            'snr_delta': row['fb_delta'],
            'bearing': bearing,
            'sin_bearing': np.sin(bearing_rad),
            'cos_bearing': np.cos(bearing_rad),
            'distance_km': distance if not np.isnan(distance) else 0,
            'front_count': row['front_spots'],
            'back_count': row['back_spots'],
            'n_paired': row['front_spots'] + row['back_spots'],
        })

    X = np.array([[p['snr_delta'], p['sin_bearing'], p['cos_bearing'], p['distance_km']] for p in points])
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    effective_min = max(2, min(3, len(points) // 2))
    clusterer = HDBSCAN(min_cluster_size=effective_min)
    labels = clusterer.fit_predict(X_scaled)
    probabilities = clusterer.probabilities_

    for i, p in enumerate(points):
        p['cluster'] = int(labels[i])
        p['membership'] = float(probabilities[i])

    cluster_points = points

    for cid in sorted(set(labels[labels >= 0])):
        members = [p for p in points if p['cluster'] == cid]
        n = len(members)
        if n < 2: continue
        deltas = np.array([p['snr_delta'] for p in members])
        memberships = np.array([p['membership'] for p in members])
        median_delta = float(np.median(deltas))
        mad = float(np.median(np.abs(deltas - median_delta)))
        mean_membership = float(np.mean(memberships))
        hl = hodges_lehmann(deltas)
        rng = np.random.default_rng(seed=42)
        boot = np.array([hodges_lehmann(rng.choice(deltas, size=n, replace=True)) for _ in range(2000)])
        ci_lo, ci_hi = float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))
        sign_positive = float(np.mean(deltas > 0))
        eps = 0.1
        signal_to_scatter = abs(median_delta) / (mad + eps)
        sign_bonus = sign_positive ** 2 if sign_positive >= 0.5 else (1.0 - sign_positive) ** 2 * 0.25
        score = math.log(max(n, 2)) * mean_membership * signal_to_scatter * sign_bonus

        cluster_scores.append({
            'cluster_id': cid, 'n_stations': n, 'hodges_lehmann': round(hl, 2),
            'bootstrap_ci_lo': round(ci_lo, 2), 'bootstrap_ci_hi': round(ci_hi, 2),
            'sign_positive': round(sign_positive, 3), 'score': round(score, 3),
            'mean_membership': round(mean_membership, 3),
            'median_delta': round(median_delta, 2), 'mad': round(mad, 2),
        })

    cluster_scores.sort(key=lambda c: c['score'], reverse=True)

    # Collect station names that belong to any cluster (not noise)
    hdbscan_stations = {p['station'] for p in points if p['cluster'] >= 0}
    noise_count = sum(1 for p in points if p['cluster'] < 0)

    print(f'\nHDBSCAN: {len(cluster_scores)} clusters, {len(hdbscan_stations)} stations in clusters, {noise_count} noise')
    if cluster_scores:
        best = cluster_scores[0]
        print(f'★ Recommended cluster {best["cluster_id"]}: HL = {best["hodges_lehmann"]:+.1f} dB '
              f'[{best["bootstrap_ci_lo"]:.1f}, {best["bootstrap_ci_hi"]:.1f}], '
              f'score = {best["score"]:.1f}')
else:
    print('\nNot enough paired stations for HDBSCAN (need ≥ 4).')

## 3 · Compute RX station coordinates

If `rx_lat` / `rx_lon` are missing, we derive approximate coordinates from the
Maidenhead grid square (`rxGrid`).  The map needs at least approximate positions.

In [ ]:
def grid_to_latlon(grid):
    """Convert Maidenhead grid square to (lat, lon) — center of square."""
    try:
        g = str(grid).strip().upper()
        if len(g) < 4: return np.nan, np.nan
        lon = (ord(g[0]) - ord('A')) * 20 - 180
        lat = (ord(g[1]) - ord('A')) * 10 - 90
        lon += int(g[2]) * 2
        lat += int(g[3]) * 1
        if len(g) >= 6:
            lon += (ord(g[4]) - ord('A')) * 2 / 24
            lat += (ord(g[5]) - ord('A')) / 24
        lon += 1.0  # center
        lat += 0.5
        return lat, lon
    except Exception:
        return np.nan, np.nan

if 'rx_lat' not in work.columns or work['rx_lat'].isna().all():
    if 'rxGrid' in work.columns:
        coords = work['rxGrid'].apply(grid_to_latlon)
        work['rx_lat'] = coords.apply(lambda c: c[0])
        work['rx_lon'] = coords.apply(lambda c: c[1])

has_coords = work['rx_lat'].notna() & work['rx_lon'].notna()
print(f'Spots with coordinates: {has_coords.sum()} / {len(work)}')

## 4 · Interactive RX Station Map — All Stations

This map shows **every RX station** that heard your beacon during the experiment.

### Reading the legend

| Color | Label | Meaning |
|-------|-------|---------|
| **Blue** | Front | Station heard you during a **front-facing run** — the antenna's main lobe was pointed toward (or near) this station's bearing |
| **Orange** | Back | Station heard you during a **back-facing run** — the antenna's back lobe / sidelobes were toward this station |
| **Red ◆** | TX | Your transmitter location |
| **Grey dotted line** | — | Connects TX to a **paired station** (heard in both orientations) |

**Important context:** Most stations on this map contribute to the *naive* (simple F/B) estimate
but are **not necessarily used by HDBSCAN**.  HDBSCAN only operates on **paired stations** —
those heard in both orientations — and further discards noise points.
The next cell shows which stations survived HDBSCAN clustering.

In [ ]:
if has_coords.any():
    geo = work[has_coords].copy()
    rxcol = 'rxCall' if 'rxCall' in geo.columns else 'rxGrid'

    stn = (geo.groupby([rxcol, 'orientation'])
           .agg(lat=('rx_lat','median'), lon=('rx_lon','median'),
                spots=('snr','count'), snr_med=('snr','median'),
                dist_med=('distance','median') if 'distance' in geo.columns else ('rx_lat','count'))
           .reset_index())

    stn['marker_size'] = np.clip(stn['spots'], 2, 10)
    stn['snr_med'] = stn['snr_med'].round(1)
    stn['dist_med'] = stn['dist_med'].round(0) if 'distance' in geo.columns else np.nan

    fig = px.scatter_geo(
        stn, lat='lat', lon='lon',
        color='orientation',
        color_discrete_map={'front': '#3b82f6', 'back': '#f97316'},
        size='marker_size',
        hover_name=rxcol,
        hover_data={'snr_med': True, 'spots': True, 'dist_med': True,
                    'marker_size': False, 'lat': False, 'lon': False},
        title='All RX Stations — Front (blue) / Back (orange)',
        labels={'snr_med': 'Median SNR (dB)', 'spots': 'Spots', 'dist_med': f'Distance ({distance_units})'},
    )

    if not np.isnan(tx_lat) and not np.isnan(tx_lon):
        fig.add_trace(go.Scattergeo(
            lat=[tx_lat], lon=[tx_lon],
            mode='markers', marker=dict(symbol='diamond', size=14, color='#dc2626'),
            name='TX', hovertemplate='TX<extra></extra>'
        ))

    proj_args = dict(projection_type='azimuthal equal area', projection_scale=0.85, showland=True, landcolor='#f3f4f6',
                     showcountries=True, countrycolor='#d1d5db',
                     showocean=True, oceancolor='#dbeafe')
    if not np.isnan(tx_lat) and not np.isnan(tx_lon):
        proj_args['projection_rotation_lat'] = tx_lat
        proj_args['projection_rotation_lon'] = tx_lon

    # Lines from TX to paired stations
    if not np.isnan(tx_lat) and not np.isnan(tx_lon):
        front_stns = set(stn.loc[stn['orientation']=='front', rxcol])
        back_stns = set(stn.loc[stn['orientation']=='back', rxcol])
        paired_calls = front_stns & back_stns
        if paired_calls:
            paired_coords = (stn[stn[rxcol].isin(paired_calls)]
                             .groupby(rxcol).agg(lat=('lat','mean'), lon=('lon','mean')).reset_index())
            for _, row in paired_coords.iterrows():
                fig.add_trace(go.Scattergeo(
                    lat=[tx_lat, row['lat']], lon=[tx_lon, row['lon']],
                    mode='lines',
                    line=dict(width=1, color='#9ca3af', dash='dot'),
                    showlegend=False, hoverinfo='skip',
                ))

    fig.update_traces(marker_sizemin=2, selector=dict(type='scattergeo'))
    fig.update_geos(**proj_args)
    fig.update_layout(width=800, height=700, margin=dict(l=40, r=40, t=50, b=20))
    fig.show()

    # Summary counts
    all_stns = stn[rxcol].nunique()
    paired_count = len(paired_calls) if not np.isnan(tx_lat) else len(pairs)
    print(f'Total unique RX stations on map: {all_stns}')
    print(f'Paired stations (grey dotted lines): {paired_count}')
    print(f'HDBSCAN cluster stations: {len(hdbscan_stations)}')
    print(f'→ {all_stns - len(hdbscan_stations)} stations are NOT used in the HDBSCAN F/B estimate')
else:
    print('No coordinates — cannot create map.')

### HDBSCAN Cluster Station Map

This map shows **only the stations that survived HDBSCAN clustering** — the ones
that actually contribute to the primary F/B estimate.

| Color | Meaning |
|-------|---------|
| **Colored circles** | Stations assigned to an HDBSCAN cluster (colored by cluster ID) |
| **Grey circles** | Noise stations — paired but excluded by HDBSCAN as outliers |
| **Red ◆** | Your TX position |

Compare this map to the one above: notice how many stations have been filtered out.
HDBSCAN identifies coherent groups of stations that share similar propagation
characteristics (signal delta, bearing, distance).  The remaining stations are
the ones providing the robust Hodges-Lehmann F/B estimate.

In [ ]:
if cluster_points and has_coords.any():
    rxcol = 'rxCall' if 'rxCall' in work.columns else 'rxGrid'
    # Build station coords lookup
    geo = work[has_coords].copy()
    stn_coords = (geo.groupby(rxcol).agg(lat=('rx_lat','median'), lon=('rx_lon','median')).reset_index())
    coords_map = {row[rxcol]: (row['lat'], row['lon']) for _, row in stn_coords.iterrows()}

    # Build cluster station dataframe
    cmap_rows = []
    recommended_id = cluster_scores[0]['cluster_id'] if cluster_scores else -1
    for p in cluster_points:
        if p['station'] in coords_map:
            lat, lon = coords_map[p['station']]
            cid = p['cluster']
            if cid >= 0:
                label = f"Cluster {cid}" + (" ★" if cid == recommended_id else "")
            else:
                label = "Noise"
            cmap_rows.append({
                'station': p['station'], 'lat': lat, 'lon': lon,
                'cluster_label': label, 'cluster': cid,
                'snr_delta': p['snr_delta'], 'membership': p['membership'],
                'distance': p['distance_km'], 'bearing': p['bearing'],
            })

    if cmap_rows:
        cdf = pd.DataFrame(cmap_rows)

        # Color map
        palette = px.colors.qualitative.Set2
        unique_cids = sorted(cdf.loc[cdf['cluster'] >= 0, 'cluster'].unique())
        color_map = {'Noise': '#d1d5db'}
        for i, cid in enumerate(unique_cids):
            label = f"Cluster {cid}" + (" ★" if cid == recommended_id else "")
            color_map[label] = palette[i % len(palette)]

        fig = px.scatter_geo(
            cdf, lat='lat', lon='lon',
            color='cluster_label', color_discrete_map=color_map,
            hover_name='station',
            hover_data={'snr_delta': ':+.1f', 'membership': ':.2f',
                        'distance': ':.0f', 'bearing': ':.0f',
                        'cluster_label': False, 'lat': False, 'lon': False},
            title='HDBSCAN Cluster Stations Only',
            labels={'snr_delta': 'F/B Delta (dB)', 'membership': 'HDBSCAN membership',
                    'distance': f'Distance ({distance_units})', 'bearing': 'Bearing (°)',
                    'cluster_label': 'Cluster'},
        )

        if not np.isnan(tx_lat) and not np.isnan(tx_lon):
            fig.add_trace(go.Scattergeo(
                lat=[tx_lat], lon=[tx_lon],
                mode='markers', marker=dict(symbol='diamond', size=14, color='#dc2626'),
                name='TX', hovertemplate='TX<extra></extra>'
            ))

            # Lines from TX to cluster stations
            for _, row in cdf[cdf['cluster'] >= 0].iterrows():
                cid = row['cluster']
                lbl = f"Cluster {cid}" + (" ★" if cid == recommended_id else "")
                fig.add_trace(go.Scattergeo(
                    lat=[tx_lat, row['lat']], lon=[tx_lon, row['lon']],
                    mode='lines',
                    line=dict(width=1.5, color=color_map.get(lbl, '#6b7280')),
                    showlegend=False, hoverinfo='skip',
                ))

        proj_args = dict(projection_type='azimuthal equal area', projection_scale=0.85,
                         showland=True, landcolor='#f3f4f6',
                         showcountries=True, countrycolor='#d1d5db',
                         showocean=True, oceancolor='#dbeafe')
        if not np.isnan(tx_lat) and not np.isnan(tx_lon):
            proj_args['projection_rotation_lat'] = tx_lat
            proj_args['projection_rotation_lon'] = tx_lon

        fig.update_traces(marker=dict(size=12, line=dict(width=1, color='white')),
                          selector=dict(type='scattergeo'))
        fig.update_geos(**proj_args)
        fig.update_layout(width=800, height=700, margin=dict(l=40, r=40, t=50, b=20))
        fig.show()

        n_cluster = len(cdf[cdf['cluster'] >= 0])
        n_noise = len(cdf[cdf['cluster'] < 0])
        print(f'Stations in clusters: {n_cluster}  |  Noise (discarded): {n_noise}')
        if cluster_scores:
            best = cluster_scores[0]
            print(f'★ Recommended cluster {best["cluster_id"]}: {best["n_stations"]} stations, '
                  f'HL = {best["hodges_lehmann"]:+.1f} dB')
    else:
        print('No cluster stations have coordinates.')
else:
    print('No HDBSCAN clusters to map (need ≥ 4 paired stations).')

## 5 · Coverage polar plot — Before and After HDBSCAN

The left chart shows bearing coverage from **all RX stations** (the naive dataset).
The right chart shows coverage from **HDBSCAN cluster stations only** — the ones
that actually contribute to the primary F/B estimate.

Comparing the two reveals which bearings survive clustering.  If entire sectors
disappear, HDBSCAN is telling you that propagation from those angles was inconsistent
or that stations there were noise.

In [ ]:
if 'bearing' in work.columns and work['bearing'].notna().any():
    rxcol = 'rxCall' if 'rxCall' in work.columns else 'rxGrid'
    bw = work[work['bearing'].notna() & work['orientation'].isin(['front','back'])].copy()
    bw['sector'] = (bw['bearing'] // 30 * 30).astype(int)

    # All stations
    all_counts = (bw.groupby(['sector','orientation'])[rxcol]
                  .nunique().reset_index().rename(columns={rxcol:'rx_count'}))

    # HDBSCAN cluster stations only
    if hdbscan_stations:
        bw_hdb = bw[bw[rxcol].isin(hdbscan_stations)].copy()
        hdb_counts = (bw_hdb.groupby(['sector','orientation'])[rxcol]
                      .nunique().reset_index().rename(columns={rxcol:'rx_count'}))
    else:
        hdb_counts = pd.DataFrame(columns=['sector','orientation','rx_count'])

    fig = make_subplots(rows=1, cols=2,
                        specs=[[{'type': 'polar'}, {'type': 'polar'}]],
                        subplot_titles=['All RX Stations', 'HDBSCAN Cluster Stations'])

    colors = {'front': '#3b82f6', 'back': '#f97316'}
    for orient in ['front', 'back']:
        # All stations (left)
        subset = all_counts[all_counts['orientation'] == orient]
        fig.add_trace(go.Barpolar(
            r=subset['rx_count'], theta=subset['sector'],
            name=f'{orient.title()} (all)', marker_color=colors[orient],
            opacity=0.7, width=25,
        ), row=1, col=1)

        # HDBSCAN cluster (right)
        subset_h = hdb_counts[hdb_counts['orientation'] == orient] if len(hdb_counts) else pd.DataFrame()
        fig.add_trace(go.Barpolar(
            r=subset_h['rx_count'] if len(subset_h) else [],
            theta=subset_h['sector'] if len(subset_h) else [],
            name=f'{orient.title()} (HDBSCAN)', marker_color=colors[orient],
            opacity=0.7, width=25,
        ), row=1, col=2)

    polar_opts = dict(angularaxis=dict(tickmode='array', tickvals=list(range(0,360,30)),
                                        direction='clockwise', rotation=90))
    fig.update_layout(height=500, polar=polar_opts, polar2=polar_opts)
    fig.show()

    all_rx = bw[rxcol].nunique()
    hdb_rx = bw_hdb[rxcol].nunique() if hdbscan_stations else 0
    print(f'All stations on polar: {all_rx}  |  HDBSCAN cluster stations: {hdb_rx}')
    if hdb_rx < all_rx and hdb_rx > 0:
        print(f'→ HDBSCAN retained {hdb_rx/all_rx*100:.0f}% of stations — the rest were noise or unpaired.')
else:
    print('No bearing data for coverage plot.')

---
## 6 · Space Weather Timeline (zoomed to experiment window)

The timeline below is **zoomed to your experiment's data collection period** with a
12-hour margin on each side, so run windows are clearly visible rather than squished
into a 7-day view.

Three key indices:
- **Kp index** — planetary geomagnetic disturbance (0–9).  Values ≥ 5 indicate a storm.
- **Solar Flux Index (SFI)** — 10.7 cm radio flux, a proxy for solar activity.
- **IMF Bz** — interplanetary magnetic field z-component.  Negative Bz = more energy entering the magnetosphere.

Shaded bands mark your front (blue) and back (orange) data collection runs.
The Kp storm threshold (dashed red line at Kp = 5) helps you see whether any runs
overlapped with elevated geomagnetic activity — this directly affects the
ionospheric confidence penalty computed in Section 9.

In [ ]:
if sw is not None and len(sw) and 'timestamp' in sw.columns:
    time_col = 'time' if 'time' in work.columns else 'timestamp_utc'
    run_times = []
    if 'run_key' in work.columns and time_col in work.columns:
        for rk, grp in work.groupby('run_key'):
            t = grp[time_col].dropna()
            if len(t):
                role = run_role_map.get(str(rk), '?')
                run_times.append((rk, t.min(), t.max(), role))

    # Compute zoom range: experiment span + 12h margin
    if run_times:
        exp_start = min(t0 for _, t0, _, _ in run_times)
        exp_end = max(t1 for _, _, t1, _ in run_times)
        margin = pd.Timedelta(hours=12)
        zoom_start = exp_start - margin
        zoom_end = exp_end + margin
    else:
        zoom_start = sw['timestamp'].min()
        zoom_end = sw['timestamp'].max()

    plots = [p for p in [('kp_index','Kp Index'), ('solar_flux','Solar Flux Index'),
                          ('imf_bz','IMF Bz (nT)')] if p[0] in sw.columns and sw[p[0]].notna().any()]

    if plots:
        fig = make_subplots(rows=len(plots), cols=1, shared_xaxes=True,
                            subplot_titles=[t for _,t in plots],
                            vertical_spacing=0.08)
        trace_colors = ['#3b82f6','#f97316','#10b981']
        for i, (col, label) in enumerate(plots, 1):
            fig.add_trace(go.Scatter(
                x=sw['timestamp'], y=sw[col],
                mode='lines+markers', name=label,
                line=dict(color=trace_colors[i-1], width=2),
                marker=dict(size=4),
            ), row=i, col=1)
            fig.update_yaxes(title_text=label, row=i, col=1)

            if col == 'kp_index':
                fig.add_hline(y=5, line_dash='dash', line_color='#dc2626',
                              annotation_text='Storm (Kp≥5)', row=i, col=1)
                fig.add_hline(y=4, line_dash='dot', line_color='#eab308',
                              annotation_text='Unsettled (Kp=4)', row=i, col=1)
            if col == 'imf_bz':
                fig.add_hline(y=0, line_color='#94a3b8', line_width=1, row=i, col=1)
                fig.add_hline(y=-5, line_dash='dot', line_color='#eab308',
                              annotation_text='Southward (Bz<-5)', row=i, col=1)

            # Run time windows
            run_colors = {'front': '#3b82f6', 'back': '#f97316'}
            for j, (rk, t0, t1, role) in enumerate(run_times):
                rc = run_colors.get(role, '#6b7280')
                fig.add_vrect(x0=t0, x1=t1, fillcolor=rc, opacity=0.15,
                              line_width=0, row=i, col=1)
                fig.add_vline(x=t0, line_dash='dot', line_color=rc, line_width=1.5, row=i, col=1)
                fig.add_vline(x=t1, line_dash='dot', line_color=rc, line_width=1.5, row=i, col=1)
                if i == 1:
                    short_rk = str(rk)[:8] if len(str(rk)) > 8 else str(rk)
                    fig.add_annotation(x=t0, y=1.0, yref='y domain',
                                       text=f'{role[0].upper()} start',
                                       showarrow=False, font=dict(size=9, color=rc), yshift=12,
                                       row=i, col=1)
                    fig.add_annotation(x=t1, y=1.0, yref='y domain',
                                       text=f'{role[0].upper()} stop',
                                       showarrow=False, font=dict(size=9, color=rc), yshift=12,
                                       row=i, col=1)

        # Zoom to experiment window
        fig.update_xaxes(range=[zoom_start, zoom_end])
        fig.update_layout(height=200*len(plots)+80, showlegend=False,
                          title_text='Space Weather during Experiment (zoomed)')
        fig.update_xaxes(title_text='UTC', row=len(plots), col=1)
        fig.show()

        # Per-run Kp overlap analysis
        if run_times and 'kp_index' in sw.columns:
            print('\n=== Kp Conditions During Each Run ===')
            kp_ts = sw[sw['kp_index'].notna()][['timestamp','kp_index']].sort_values('timestamp')
            for rk, t0, t1, role in run_times:
                window = kp_ts[(kp_ts['timestamp'] >= t0) & (kp_ts['timestamp'] <= t1)]
                if len(window):
                    kp_vals = window['kp_index']
                    max_kp = kp_vals.max()
                    mean_kp = kp_vals.mean()
                    flag = ''
                    if max_kp >= 6: flag = ' ✗ STORM'
                    elif max_kp >= 5: flag = ' ⚠ MINOR STORM'
                    elif max_kp >= 4: flag = ' ⚠ UNSETTLED'
                    else: flag = ' ✓ QUIET'
                    short_rk = str(rk)[:8]
                    print(f'  Run {short_rk}… ({role}): Kp mean={mean_kp:.1f}, max={max_kp:.0f}{flag}')
                else:
                    # Find nearest Kp reading
                    mid = t0 + (t1 - t0) / 2
                    diffs = abs(kp_ts['timestamp'] - mid)
                    if len(diffs):
                        nearest = kp_ts.iloc[diffs.argmin()]
                        print(f'  Run {str(rk)[:8]}… ({role}): nearest Kp = {nearest["kp_index"]:.0f} '
                              f'(no readings within run window)')
    else:
        print('Space weather CSV exists but has no plottable numeric columns.')
else:
    print('No space weather data — skipping timeline.')

## 7 · Naive SNR estimate vs Kp correlation

This analysis uses **all stations** (not HDBSCAN-filtered) to show how raw SNR
varies with geomagnetic activity.  This is deliberately the *naive* approach:
it ignores station pairing, HDBSCAN clustering, and propagation mode differences.

The value of this chart is diagnostic: if SNR drops noticeably at higher Kp values,
it confirms that geomagnetic conditions affected your data.  This justifies the
ionospheric confidence penalty applied in Section 9.

In [ ]:
if sw is not None and 'kp_index' in sw.columns and sw['kp_index'].notna().any():
    time_col = 'time' if 'time' in work.columns else 'timestamp_utc'
    if time_col in work.columns and work[time_col].notna().any():
        kp_series = sw[sw['kp_index'].notna()][['timestamp','kp_index']].sort_values('timestamp').reset_index(drop=True)
        kp_timestamps = kp_series['timestamp'].values.astype('datetime64[ns]')
        spots_t = work[work['snr'].notna() & work[time_col].notna()].copy()

        spot_times = spots_t[time_col].values.astype('datetime64[ns]')
        idxs = np.searchsorted(kp_timestamps, spot_times)
        idxs = np.clip(idxs, 0, len(kp_series) - 1)
        spots_t['kp_nearest'] = kp_series['kp_index'].iloc[idxs].values

        fig = px.box(spots_t, x='kp_nearest', y='snr', color='orientation',
                     color_discrete_map={'front':'#3b82f6','back':'#f97316'},
                     title='Naive SNR vs Kp (all stations, no HDBSCAN filtering)',
                     labels={'kp_nearest':'Kp Index','snr':'SNR (dB)'})
        fig.update_layout(height=400)
        fig.show()

        kp_summary = (spots_t.groupby(['kp_nearest','orientation'])['snr']
                      .agg(['median','count'])
                      .reset_index()
                      .rename(columns={'median':'SNR median','count':'Spots'}))
        display(kp_summary)
    else:
        print('No timestamped spots — cannot correlate with Kp.')
else:
    print('No Kp data — skipping correlation.')

## 8 · Space weather summary for this experiment

A quick summary of conditions during your data collection, useful for the experiment report.

In [ ]:
if sw is not None and len(sw):
    # Narrow to experiment window if possible
    time_col = 'time' if 'time' in work.columns else 'timestamp_utc'
    if time_col in work.columns and work[time_col].notna().any():
        exp_start = work[time_col].min()
        exp_end = work[time_col].max()
        margin = pd.Timedelta(hours=3)
        sw_window = sw[(sw['timestamp'] >= exp_start - margin) & (sw['timestamp'] <= exp_end + margin)]
        window_label = f'experiment window (±3h)'
    else:
        sw_window = sw
        window_label = 'full 7-day window'

    sw_display = sw_window if len(sw_window) >= 3 else sw

    print(f'Space weather summary ({window_label}):')
    print(f'  Time range: {sw_display["timestamp"].min()} – {sw_display["timestamp"].max()}')
    for col, label, unit in [('kp_index','Kp',''), ('solar_flux','SFI','SFU'),
                              ('sunspot_number','Sunspot #',''),
                              ('imf_bz','IMF Bz','nT')]:
        if col in sw_display.columns and sw_display[col].notna().any():
            vals = sw_display[col].dropna()
            print(f'  {label}: min={vals.min():.1f} mean={vals.mean():.1f} max={vals.max():.1f} {unit}')
    if 'xray_class' in sw_display.columns:
        classes = sw_display['xray_class'].dropna().unique()
        if len(classes):
            print(f'  X-ray classes seen: {", ".join(str(c) for c in classes[:10])}')
    if 'kp_index' in sw_display.columns:
        max_kp = sw_display['kp_index'].max()
        if max_kp >= 7: condition = 'Severe storm (Kp≥7)'
        elif max_kp >= 5: condition = 'Minor storm (Kp≥5)'
        elif max_kp >= 4: condition = 'Unsettled (Kp=4)'
        else: condition = 'Quiet (Kp<4)'
        print(f'  Geomagnetic conditions: {condition}')
else:
    print('No space weather data.')

## 9 · Ionospheric stability confidence penalty — applied to the HDBSCAN estimate

The confidence score from Part 2 measures *data quality* — how many stations, how tight
the CI, how consistent the sign.  But even excellent data can be misleading if the
ionosphere was disturbed during collection.

### The formula

The **ionospheric stability penalty** adjusts the base confidence multiplicatively:

$$\text{final\_confidence} = \text{base\_confidence} \times \text{iono\_penalty}$$

where the penalty is computed from two indices:

| Index | Weight | Factor computation |
|-------|--------|--------------------|
| **Kp** | 60% | 1.0 if Kp < 3, linear decay to 0 at Kp = 9 |
| **IMF Bz** | 40% | 1.0 if Bz > 0, linear decay to 0 at Bz = −20 nT |

$$\text{iono\_stability} = 0.6 \times \text{kp\_factor} + 0.4 \times \text{bz\_factor}$$
$$\text{iono\_penalty} = 0.7 + 0.3 \times \text{iono\_stability}$$

The penalty ranges from **0.70** (worst: storm + strongly negative Bz) to **1.00** (quiet).

Principle: **"Cluster first, condition second."**  HDBSCAN clustering operates on signal
differences only — space weather is a *quality gate*, not a clustering feature.

Below, we compute the penalty and apply it step-by-step to the actual HDBSCAN
Hodges-Lehmann estimate from this experiment.

In [ ]:
# --- Step 1: Compute the penalty factors ---
if sw is not None and len(sw):
    # Use experiment-windowed Kp and Bz
    time_col = 'time' if 'time' in work.columns else 'timestamp_utc'
    if time_col in work.columns and work[time_col].notna().any():
        exp_start = work[time_col].min()
        exp_end = work[time_col].max()
        margin = pd.Timedelta(hours=3)
        sw_exp = sw[(sw['timestamp'] >= exp_start - margin) & (sw['timestamp'] <= exp_end + margin)]
    else:
        sw_exp = sw

    sw_used = sw_exp if len(sw_exp) >= 3 else sw

    mean_kp = sw_used['kp_index'].mean() if 'kp_index' in sw_used.columns and sw_used['kp_index'].notna().any() else 0
    kp_factor = max(0, min(1, (9 - mean_kp) / 6)) if mean_kp >= 3 else 1.0

    mean_bz = sw_used['imf_bz'].mean() if 'imf_bz' in sw_used.columns and sw_used['imf_bz'].notna().any() else 0
    bz_factor = max(0, min(1, (mean_bz + 20) / 20)) if mean_bz < 0 else 1.0

    iono_stability = 0.6 * kp_factor + 0.4 * bz_factor
    iono_penalty = 0.7 + 0.3 * iono_stability

    print('╔══════════════════════════════════════════════════════════╗')
    print('║     IONOSPHERIC STABILITY PENALTY — STEP BY STEP       ║')
    print('╚══════════════════════════════════════════════════════════╝')
    print()
    print('Step 1: Input indices (experiment window)')
    print(f'  Mean Kp  = {mean_kp:.2f}')
    print(f'  Mean Bz  = {mean_bz:.2f} nT')
    print()

    print('Step 2: Compute factors')
    print(f'  kp_factor = max(0, min(1, (9 - {mean_kp:.2f}) / 6)) = {kp_factor:.3f}')
    print(f'  bz_factor = max(0, min(1, ({mean_bz:.2f} + 20) / 20)) = {bz_factor:.3f}')
    print()

    print('Step 3: Combine into stability and penalty')
    print(f'  iono_stability = 0.6 × {kp_factor:.3f} + 0.4 × {bz_factor:.3f} = {iono_stability:.3f}')
    print(f'  iono_penalty   = 0.7 + 0.3 × {iono_stability:.3f} = {iono_penalty:.3f}')
    print()

    if iono_penalty >= 0.95:
        print('  ✓ Quiet conditions — no significant penalty.')
    elif iono_penalty >= 0.85:
        print(f'  ⚠ Mildly disturbed — confidence reduced by {(1-iono_penalty)*100:.1f}%.')
    else:
        print(f'  ✗ Disturbed conditions — confidence reduced by {(1-iono_penalty)*100:.0f}%.')
        print('    Consider repeating the experiment under quieter conditions.')

    # --- Step 4: Quality gate checks ---
    print()
    max_kp = sw_used['kp_index'].max() if 'kp_index' in sw_used.columns else 0
    std_kp = sw_used['kp_index'].std() if 'kp_index' in sw_used.columns else 0
    min_bz = sw_used['imf_bz'].min() if 'imf_bz' in sw_used.columns else 0

    print('Step 4: Space weather quality gates')
    if max_kp >= 6:
        print(f'  ✗ FAIL  Kp reached {max_kp:.0f} (≥6 = strong storm during experiment)')
    elif max_kp >= 5 or std_kp > 1.5:
        reason = []
        if max_kp >= 5: reason.append(f'Kp max={max_kp:.0f}')
        if std_kp > 1.5: reason.append(f'Kp σ={std_kp:.1f}')
        print(f'  ⚠ WARN  Kp stability: {", ".join(reason)}')
    else:
        print(f'  ✓ PASS  Kp stability (max={max_kp:.0f}, σ={std_kp:.1f})')

    if min_bz <= -10:
        print(f'  ✗ FAIL  IMF Bz reached {min_bz:.1f} nT (≤−10 = storm driving)')
    elif mean_bz < -5:
        print(f'  ⚠ WARN  IMF Bz mean = {mean_bz:.1f} nT (< −5 = mildly southward)')
    else:
        print(f'  ✓ PASS  IMF Bz (mean={mean_bz:.1f} nT, min={min_bz:.1f} nT)')

    # --- Step 5: Apply to HDBSCAN estimate ---
    print()
    if cluster_scores:
        best = cluster_scores[0]
        hl = best['hodges_lehmann']
        ci_lo = best['bootstrap_ci_lo']
        ci_hi = best['bootstrap_ci_hi']

        # Recompute base confidence from Part 2 logic
        n_pairs = len(pairs)
        total_spots = len(work[work['snr'].notna()])
        front_spots = len(work[(work['orientation']=='front') & work['snr'].notna()])
        back_spots = len(work[(work['orientation']=='back') & work['snr'].notna()])
        classified_spots = front_spots + back_spots
        total_rx = work['rxCall'].nunique() if 'rxCall' in work.columns else 1
        evidence = min(1, n_pairs / 15)
        classified_ratio = classified_spots / max(total_spots, 1)
        overlap_ratio = n_pairs / max(total_rx, 1)
        cluster_quality = min(1, best['score'] / 5.0)
        ci_width = ci_hi - ci_lo
        ci_tightness = max(0, 1 - ci_width / 10)
        base_conf = 0.30 * evidence + 0.15 * classified_ratio + 0.15 * overlap_ratio + 0.25 * cluster_quality + 0.15 * ci_tightness
        final_conf = base_conf * iono_penalty

        if final_conf >= 0.72: conf_class = 'High'
        elif final_conf >= 0.45: conf_class = 'Moderate'
        else: conf_class = 'Low'

        print('Step 5: Apply penalty to the HDBSCAN F/B estimate')
        print()
        print(f'  ★ HDBSCAN Cluster {best["cluster_id"]} F/B estimate:')
        print(f'    Hodges-Lehmann    = {hl:+.1f} dB')
        print(f'    95% Bootstrap CI  = [{ci_lo:.1f}, {ci_hi:.1f}] dB')
        print(f'    Sign+             = {best["sign_positive"]*100:.0f}%')
        print(f'    Cluster score     = {best["score"]:.1f}')
        print()
        print(f'  Base confidence (from Part 2):')
        print(f'    = 0.30×{evidence:.2f} + 0.15×{classified_ratio:.2f} + 0.15×{overlap_ratio:.2f} '
              f'+ 0.25×{cluster_quality:.2f} + 0.15×{ci_tightness:.2f}')
        print(f'    = {base_conf:.3f} ({base_conf*100:.0f}%)')
        print()
        print(f'  Final confidence (with iono penalty):')
        print(f'    = {base_conf:.3f} × {iono_penalty:.3f}')
        print(f'    = {final_conf:.3f} ({final_conf*100:.0f}%) → {conf_class}')
        print()
        print('  ┌─────────────────────────────────────────────────────┐')
        print(f'  │  F/B = {hl:+.1f} dB [{ci_lo:.1f}, {ci_hi:.1f}]')
        print(f'  │  Confidence = {final_conf*100:.0f}% ({conf_class})')
        if iono_penalty < 0.95:
            print(f'  │  ⚠ Reduced from {base_conf*100:.0f}% by {(1-iono_penalty)*100:.1f}% iono penalty')
        print('  └─────────────────────────────────────────────────────┘')
    else:
        print('Step 5: No HDBSCAN clusters — penalty applies to paired median instead.')
        if len(pairs):
            fb_med = pairs['fb_delta'].median()
            print(f'  Paired median F/B = {fb_med:+.1f} dB')
            print(f'  iono_penalty = {iono_penalty:.3f}')
            print(f'  (Penalty adjusts confidence score only, not the F/B value itself.)')

else:
    iono_penalty = 1.0
    print('No space weather data — iono_penalty = 1.00 (no adjustment).')
    if cluster_scores:
        best = cluster_scores[0]
        print(f'\n★ HDBSCAN F/B = {best["hodges_lehmann"]:+.1f} dB — no penalty applied.')

---

## Wrapping up

**You've now completed the full analysis pipeline:**

1. **Part 1** — Data quality, distributions, and overview
2. **Part 2** — Front/back classification, paired F/B estimation, HDBSCAN cluster analysis,
   Hodges-Lehmann estimator, bootstrap confidence intervals, and composite cluster scoring
3. **Part 3** — Geographic coverage (all stations vs HDBSCAN), space weather context,
   Kp overlap with runs, and ionospheric stability confidence penalty applied to the actual estimate

### Key metrics for your report

| Metric | Source |
|--------|--------|
| **HDBSCAN cluster F/B** (Hodges-Lehmann) with 95% CI | Part 2, Section 6 |
| Station-paired F/B median | Part 2, Section 5 |
| Simple sector F/B (weakest) | Part 2, Section 7 |
| Base confidence score | Part 2, Section 10 |
| Final confidence (base × iono_penalty) | Part 3, Section 9 |
| Quality gate status | Parts 2 & 3 |
| Geomagnetic conditions | Part 3, Sections 6–8 |

Save your notebooks as HTML (`File → Save and Export Notebook As → HTML`) for your records.